# Full Experiment

运行完整实验并导出结果到 `report/results/`。

In [1]:
from pathlib import Path

import pandas as pd

from src.experiment import run_experiment

In [2]:
ROOT = Path.cwd()
RESULT_DIR = ROOT / 'report' / 'results'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

CSV_OUT = RESULT_DIR / 'hf_80_metrics_filtered.csv'
PLOT_OUT = RESULT_DIR / 'hf_80_structure_filtered.pdf'
CORR_OUT = RESULT_DIR / 'hf_80_correlations_filtered.csv'

TEXT_DATASET = 'wikitext'
TEXT_CONFIG = 'wikitext-2-raw-v1'
TEXT_COLUMN = 'text'
IMAGE_DATASET = 'mnist'
AUDIO_DATASET = 'PolyAI/minds14'
AUDIO_CONFIG = 'en-US'
AUDIO_COLUMN = 'audio'
HF_SPLIT = 'train[:400]'
LIMIT = 80
MIN_TEXT_BYTES = 200

print('Result directory:', RESULT_DIR)
print('CSV output     :', CSV_OUT)
print('Plot output    :', PLOT_OUT)
print('Corr output    :', CORR_OUT)

Result directory: /Users/nanjihuaji/code/mediaexp/report/results
CSV output     : /Users/nanjihuaji/code/mediaexp/report/results/hf_80_metrics_filtered.csv
Plot output    : /Users/nanjihuaji/code/mediaexp/report/results/hf_80_structure_filtered.pdf
Corr output    : /Users/nanjihuaji/code/mediaexp/report/results/hf_80_correlations_filtered.csv


In [3]:
rows = run_experiment(
    hf_text_dataset=TEXT_DATASET,
    hf_text_config_name=TEXT_CONFIG,
    hf_text_column=TEXT_COLUMN,
    hf_image_dataset=IMAGE_DATASET,
    hf_audio_dataset=AUDIO_DATASET,
    hf_audio_config_name=AUDIO_CONFIG,
    hf_audio_column=AUDIO_COLUMN,
    hf_split=HF_SPLIT,
    limit_per_dataset=LIMIT,
    min_text_bytes=MIN_TEXT_BYTES,
    csv_out=CSV_OUT,
    plot_out=PLOT_OUT,
    corr_out=CORR_OUT,
)

len(rows)

/Users/nanjihuaji/code/mediaexp/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Running compression experiment: 100%|██████████| 240/240 [06:34<00:00,  1.64s/sample] 


category | dataset        | name        | input_bytes | structure_score | lzw_ratio | lzss_ratio | detail                       | source                      
---------+----------------+-------------+-------------+-----------------+-----------+------------+------------------------------+-----------------------------
text     | wikitext       | sample_0003 | 727         | 0.4727          | 0.77      | 0.865      | HF text, 2 lines, 727 bytes  | <memory>                    
text     | wikitext       | sample_0004 | 524         | 0.4946          | 0.818     | 0.896      | HF text, 2 lines, 524 bytes  | <memory>                    
text     | wikitext       | sample_0005 | 574         | 0.5011          | 0.784     | 0.836      | HF text, 2 lines, 574 bytes  | <memory>                    
text     | wikitext       | sample_0009 | 1221        | 0.5036          | 0.66      | 0.74       | HF text, 2 lines, 1221 bytes | <memory>                    
text     | wikitext       | sample_0010 | 1623

240

In [4]:
metrics_df = pd.read_csv(CSV_OUT)
corr_df = pd.read_csv(CORR_OUT)

display(metrics_df.head())
display(corr_df)

,category,dataset,name,source,input_bytes,lzw_units,lzss_units,structure_score,byte_entropy,window_entropy,lzw_ratio,lzss_ratio,detail
0,text,wikitext,sample_0003,<memory>,727,448,468,0.4727,4.6865,7.5017,0.770,0.865,"HF text, 2 lines, 727 bytes"
1,text,wikitext,sample_0004,<memory>,524,343,364,0.4946,4.4091,7.3533,0.818,0.896,"HF text, 2 lines, 524 bytes"
2,text,wikitext,sample_0005,<memory>,574,360,359,0.5011,4.3725,7.2190,0.784,0.836,"HF text, 2 lines, 574 bytes"
3,text,wikitext,sample_0009,<memory>,1221,645,618,0.5036,4.2845,7.3169,0.660,0.740,"HF text, 2 lines, 1221 bytes"
4,text,wikitext,sample_0010,<memory>,1623,810,746,0.4973,4.3362,7.4139,0.624,0.691,"HF text, 2 lines, 1623 bytes"


,metric,correlation,slope,intercept
0,LZW,-0.97751,-1.252202,1.306318
1,LZSS,-0.91192,-1.009177,1.204058


In [ ]:
summary_df = metrics_df.groupby('category')[['structure_score', 'lzw_ratio', 'lzss_ratio']].agg(['mean', 'std'])
summary_df

## Regression Analysis

按全部样本和各数据类型分别计算线性回归结果，输出斜率、截距、相关系数 `r`、`p` 值和 `t` 统计量。

In [ ]:
from scipy.stats import linregress

regression_rows = []
for category, group_df in [('all', metrics_df)] + list(metrics_df.groupby('category')):
    subset = group_df[group_df['input_bytes'] > 0]
    for metric in ['lzw_ratio', 'lzss_ratio']:
        result = linregress(subset['structure_score'], subset[metric])
        t_stat = result.slope / result.stderr if result.stderr not in (0, None) else float('nan')
        regression_rows.append({
            'category': category,
            'metric': metric,
            'n': len(subset),
            'slope': result.slope,
            'intercept': result.intercept,
            'r': result.rvalue,
            'p': result.pvalue,
            't': t_stat,
        })

regression_df = pd.DataFrame(regression_rows)
regression_df

## Regression Plots

绘制各数据类型及全部样本的结构评分与压缩率散点图及线性拟合结果。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=True, sharey=True)
axes = axes.flatten()
groups = [('text', 'Text'), ('image', 'Image'), ('audio', 'Audio'), ('all', 'All Samples')]

for ax, (category, title) in zip(axes, groups):
    subset = metrics_df if category == 'all' else metrics_df[metrics_df['category'] == category]
    subset = subset[subset['input_bytes'] > 0]
    x = subset['structure_score'].to_numpy()
    ax.scatter(x, subset['lzw_ratio'], s=20, alpha=0.7, color='#1f2937', label='LZW samples')
    ax.scatter(x, subset['lzss_ratio'], s=20, alpha=0.7, color='#6b7280', label='LZSS samples')

    for metric, color, label in [('lzw_ratio', '#2563eb', 'LZW fit'), ('lzss_ratio', '#dc2626', 'LZSS fit')]:
        row = regression_df[(regression_df['category'] == category) & (regression_df['metric'] == metric)]
        if row.empty:
            continue
        slope = row.iloc[0]['slope']
        intercept = row.iloc[0]['intercept']
        r_value = row.iloc[0]['r']
        x_line = np.linspace(0, 1, 200)
        y_line = slope * x_line + intercept
        ax.plot(x_line, y_line, color=color, linewidth=2, label=f'{label} (r={r_value:.3f})')

    ax.set_title(title)
    ax.set_xlim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best', fontsize=9)

axes[2].set_xlabel('Structure Score')
axes[3].set_xlabel('Structure Score')
axes[0].set_ylabel('Compression Ratio')
axes[2].set_ylabel('Compression Ratio')
plt.tight_layout()
plt.show()